# Flickr Signal Validation

Validates geotagged Flickr photo upload rates as a leading indicator of real estate price booms.

For each location:
- Plot raw quarterly photo count vs price inflection year
- Plot rolling-smoothed count + z-score
- Plot unique user count (discovery diversity)
- Plot discovery ratio (unique users / total photos)
- Summarise lead times across all 10 locations

In [ ]:
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

CONFIG_PATH   = Path('../config/locations.yaml')
PROCESSED_DIR = Path('../data/processed/flickr')

with open(CONFIG_PATH) as f:
    locations = yaml.safe_load(f)['locations']

print(f'Loaded {len(locations)} locations')

In [ ]:
def load_features(loc_id: str) -> pd.DataFrame | None:
    path = PROCESSED_DIR / f'{loc_id}_flickr_features.csv'
    if not path.exists():
        print(f'  ⚠️  No processed data for {loc_id}')
        return None
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    return df


def plot_location(loc_id: str, loc_name: str, inflection_year: int):
    df = load_features(loc_id)
    if df is None:
        return

    fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)
    fig.suptitle(f'{loc_name} — Flickr signal vs price inflection ({inflection_year})',
                 fontsize=14, fontweight='bold')

    inflection_line = pd.Timestamp(f'{inflection_year}-01-01')
    vline_kw = dict(color='red', linestyle='--', linewidth=1.5, label=f'Price inflection ({inflection_year})')

    # ── Panel 1: Raw photo count ──────────────────────────────────────────
    ax = axes[0]
    ax.bar(df.index, df['photo_count'], width=80, color='steelblue', alpha=0.6, label='Photos/quarter')
    if 'photo_count_roll4q' in df.columns:
        ax.plot(df.index, df['photo_count_roll4q'], color='navy', linewidth=2, label='4-quarter rolling avg')
    ax.axvline(inflection_line, **vline_kw)
    ax.set_ylabel('Photo count')
    ax.set_title('Geotagged photos per quarter')
    ax.legend(fontsize=8)

    # ── Panel 2: Unique users ─────────────────────────────────────────────
    ax = axes[1]
    ax.bar(df.index, df['unique_users'], width=80, color='coral', alpha=0.6, label='Unique photographers')
    if 'unique_users_roll4q' in df.columns:
        ax.plot(df.index, df['unique_users_roll4q'], color='firebrick', linewidth=2, label='4-quarter rolling avg')
    ax.axvline(inflection_line, **vline_kw)
    ax.set_ylabel('Unique users')
    ax.set_title('Unique photographers per quarter (discovery diversity)')
    ax.legend(fontsize=8)

    # ── Panel 3: Discovery ratio ──────────────────────────────────────────
    ax = axes[2]
    if 'discovery_ratio' in df.columns:
        ax.plot(df.index, df['discovery_ratio'], color='purple', linewidth=1, alpha=0.5, label='Raw ratio')
    if 'discovery_ratio_roll4q' in df.columns:
        ax.plot(df.index, df['discovery_ratio_roll4q'], color='purple', linewidth=2, label='4-quarter rolling')
    ax.axvline(inflection_line, **vline_kw)
    ax.set_ylabel('Ratio')
    ax.set_title('Discovery ratio (unique users / photos) — high = many new visitors')
    ax.legend(fontsize=8)

    # ── Panel 4: Z-score ──────────────────────────────────────────────────
    ax = axes[3]
    z_col = 'photo_count_roll4q_zscore'
    if z_col in df.columns:
        ax.plot(df.index, df[z_col], color='green', linewidth=2, label='Z-score (smoothed count)')
        ax.axhline(2.0, color='orange', linestyle=':', linewidth=1.5, label='Z=2 threshold')
        ax.fill_between(df.index, 2.0, df[z_col].clip(lower=2.0),
                        alpha=0.2, color='orange', label='Above threshold')
    ax.axvline(inflection_line, **vline_kw)
    ax.set_ylabel('Z-score')
    ax.set_title('Standardised interest vs 2005–2014 baseline')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(12))

    plt.tight_layout()
    out_path = PROCESSED_DIR / f'{loc_id}_flickr_validation.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → {out_path}')

In [ ]:
# Plot all locations
for loc in locations:
    plot_location(loc['id'], loc['name'], loc['price_inflection_year'])

In [ ]:
# Cross-location lead time summary
summary_path = PROCESSED_DIR / '_signal_summary.csv'

if summary_path.exists():
    summary = pd.read_csv(summary_path)

    print('\n── Flickr signal lead times ───────────────────────────────────')
    display_cols = [
        'location_name', 'price_inflection_year',
        'best_break_year', 'z_fire_year',
        'lead_time_break_yrs', 'lead_time_z_yrs',
        'total_photos_all_time',
    ]
    print(summary[display_cols].to_string(index=False))

    # Summary stats
    valid_leads = summary['lead_time_z_yrs'].dropna()
    if len(valid_leads) > 0:
        print(f'\nMedian lead time (z-score method): {valid_leads.median():.1f} years')
        print(f'Mean lead time:                    {valid_leads.mean():.1f} years')
        print(f'Range:                             {valid_leads.min():.0f} – {valid_leads.max():.0f} years')
else:
    print('No summary file found — run 04_flickr_process.py first')

In [ ]:
# Compare Flickr lead time vs Google Trends lead time (if available)
gt_summary_path = Path('../data/processed/google_trends/_signal_summary.csv')
flickr_summary_path = PROCESSED_DIR / '_signal_summary.csv'

if gt_summary_path.exists() and flickr_summary_path.exists():
    gt      = pd.read_csv(gt_summary_path)
    flickr  = pd.read_csv(flickr_summary_path)

    # Merge on location_id
    merged = flickr[['location_id', 'location_name', 'price_inflection_year',
                     'lead_time_z_yrs']].rename(columns={'lead_time_z_yrs': 'flickr_lead_yrs'})

    # Pull GT 'awareness' lead time if it exists in GT summary
    if 'lead_time_years' in gt.columns and 'query_type' in gt.columns:
        gt_awareness = gt[gt['query_type'] == 'awareness'][['location_id', 'lead_time_years']]
        gt_awareness = gt_awareness.rename(columns={'lead_time_years': 'gt_awareness_lead_yrs'})
        merged = merged.merge(gt_awareness, on='location_id', how='left')

    print('\n── Flickr vs Google Trends lead time comparison ───────────────')
    print(merged.to_string(index=False))
else:
    print('Run both Google Trends and Flickr pipelines to enable comparison.')